# PDDL Attack Path Generation
Generates PDDL attack paths from CVE natural language descriptions using the `cve2pddlap` pipeline.

**Run order:** Cell 1 → 2 → 3(interactive preview) → 4 (LLM generation)

## 1. Environment Setup

In [11]:
import sys
import os

sys.path.insert(0, '../../src')

from cve2pddlap.core.data_loader import load_cve_list, load_few_shot_pool, select_few_shot_examples, FewShotExample
from cve2pddlap.core.prompt_builder import build_messages, ALL_SELECTABLE_NAMES
from cve2pddlap.core.experiment_runner import run_single, run_batch
from cve2pddlap.llm_providers import create_provider
from cve2pddlap.utils.config import get_api_key, settings

print('Dependencies loaded successfully')
print(f'Selectable instructions: {ALL_SELECTABLE_NAMES}')

Dependencies loaded successfully
Selectable instructions: ['SMI1', 'SMI2', 'SMI3', 'SMI4', 'SMI5', 'OI1', 'OI2', 'OI3']


## 2. Data

In [16]:
DATASET_PATH = '../../resources/data/CVE-PDDL-NNL-ReAP'
TARGET_POOL_PATH = '../../resources/data/target_pool.json'

# Reference dataset: all CVE/AP examples available for few-shot
few_shot_pool = load_few_shot_pool(DATASET_PATH)
pool_by_key = {ex.key: ex for ex in few_shot_pool}
print(f'Few-shot pool: {len(few_shot_pool)} examples ({len(set(ex.cve_id for ex in few_shot_pool))} CVEs)')

# Target pool: CVEs to generate attack paths for (edit target_pool.json to customise)
target_pool = load_cve_list(TARGET_POOL_PATH)
target_by_id = {entry.cve_id: entry for entry in target_pool}
target_cve_ids = [entry.cve_id for entry in target_pool]
print(f'Target pool: {len(target_pool)} CVEs')
for entry in target_pool:
    print(f'  {entry.cve_id}')

Few-shot pool: 55 examples (21 CVEs)
Target pool: 26 CVEs
  CVE-2025-66032
  CVE-2025-64755
  CVE-2025-59536
  CVE-2025-54795
  CVE-2025-9930
  CVE-2022-1471
  CVE-2022-40149
  CVE-2022-40150
  CVE-2023-2976
  CVE-2023-33202
  CVE-2023-34055
  CVE-2023-44487
  CVE-2023-46589
  CVE-2023-6378
  CVE-2024-12798
  CVE-2024-22243
  CVE-2024-22259
  CVE-2024-22262
  CVE-2024-34447
  CVE-2024-38286
  CVE-2024-38809
  CVE-2024-38816
  CVE-2024-38820
  CVE-2024-47072
  CVE-2025-22228
  CVE-2025-24813


## 3. Interactive Prompt Preview
Select CVE, instructions, few-shot count, and specific few-shot examples to preview the rendered prompt.

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Target CVE (from target_pool.json) ---
cve_dropdown = widgets.Dropdown(
    options=target_cve_ids,
    value=target_cve_ids[0] if target_cve_ids else None,
    description='Target CVE:',
    layout=widgets.Layout(width='380px')
)

# --- Optional instructions ---
oi_select = widgets.SelectMultiple(
    options=ALL_SELECTABLE_NAMES,
    description='Instructions:',
    rows=8,
    layout=widgets.Layout(width='240px')
)

# --- Few-shot count slider ---
few_shot_slider = widgets.IntSlider(
    value=0, min=0, max=5, step=1,
    description='Few-shot:',
    layout=widgets.Layout(width='380px')
)

def _example_options(exclude_cve):
    """All CVE/AP keys except those belonging to exclude_cve."""
    return ['(random)'] + [k for k in sorted(pool_by_key) if not k.startswith(exclude_cve)]

# --- Dynamic few-shot example selectors (up to 5) ---
example_selectors = []
for i in range(5):
    sel = widgets.Dropdown(
        options=_example_options(cve_dropdown.value),
        value='(random)',
        description=f'Example {i+1}:',
        layout=widgets.Layout(width='380px'),
        disabled=True
    )
    example_selectors.append(sel)

example_box = widgets.VBox(example_selectors)

def on_slider_change(change):
    n = change['new']
    opts = _example_options(cve_dropdown.value)
    for i, sel in enumerate(example_selectors):
        sel.disabled = (i >= n)
        sel.options = opts
        if i >= n:
            sel.value = '(random)'

def on_cve_change(change):
    opts = _example_options(change['new'])
    for sel in example_selectors:
        sel.options = opts
        sel.value = '(random)'

few_shot_slider.observe(on_slider_change, names='value')
cve_dropdown.observe(on_cve_change, names='value')

# --- Preview button ---
preview_button = widgets.Button(
    description='Preview Prompt',
    button_style='info',
    layout=widgets.Layout(width='160px')
)
output_area = widgets.Output()

def on_preview(b):
    with output_area:
        clear_output()
        cve_id = cve_dropdown.value
        selected_oi = list(oi_select.value)
        n_shot = few_shot_slider.value

        entry = target_by_id[cve_id]

        examples = None
        if n_shot > 0:
            chosen = []
            random_slots = []
            for i in range(n_shot):
                val = example_selectors[i].value
                if val != '(random)':
                    chosen.append(pool_by_key[val])
                else:
                    random_slots.append(i)
            if random_slots:
                exclude = {cve_id} | {ex.cve_id for ex in chosen}
                rand_pool = [ex for ex in few_shot_pool if ex.cve_id not in exclude]
                import random as _random
                rng = _random.Random(42)
                rand_picks = rng.sample(rand_pool, min(len(random_slots), len(rand_pool)))
                chosen.extend(rand_picks)
            examples = chosen[:n_shot]
            print(f'Few-shot examples: {[e.key for e in examples]}\n')

        messages = build_messages(
            cve_id=cve_id,
            cve_description=entry.description,
            selected_oi=selected_oi,
            few_shot_examples=examples
        )

        print(f'CVE: {cve_id} | Instructions: {selected_oi or ["mandatory only"]} | {n_shot}-shot')
        print(f'Messages: {len(messages)}\n')
        for msg in messages:
            print(f"{'='*20} [{msg['role'].upper()}] {'='*20}")
            print(msg['content'])
            print()

preview_button.on_click(on_preview)

display(
    widgets.HBox([
        widgets.VBox([
            cve_dropdown,
            few_shot_slider,
            widgets.Label('Drag slider to enable example selectors:'),
            example_box,
            preview_button
        ]),
        oi_select
    ]),
    output_area
)

Output()

## 4. LLM Generation
Run this cell only after verifying the prompt in Cell 4.

In [14]:
# Check API key
try:
    key = get_api_key('qwen')
    print(f'Qwen API key loaded: {key[:8]}...')
except ValueError as e:
    print(f'Error: {e}')
    print('Please check QWEN_API_KEY in .env')

Qwen API key loaded: sk-717c6...


In [18]:
# Read configuration directly from the widgets above
TARGET_CVE = cve_dropdown.value
SELECTED_OI = list(oi_select.value)
N_SHOT = few_shot_slider.value
SAVE_OUTPUT = True

entry = target_by_id[TARGET_CVE]

# Resolve examples from selectors (same logic as Preview)
examples = None
if N_SHOT > 0:
    chosen = []
    random_slots = []
    for i in range(N_SHOT):
        val = example_selectors[i].value
        if val != '(random)':
            chosen.append(pool_by_key[val])
        else:
            random_slots.append(i)
    if random_slots:
        exclude = {TARGET_CVE} | {ex.cve_id for ex in chosen}
        rand_pool = [ex for ex in few_shot_pool if ex.cve_id not in exclude]
        import random as _random
        rng = _random.Random(42)
        rand_picks = rng.sample(rand_pool, min(len(random_slots), len(rand_pool)))
        chosen.extend(rand_picks)
    examples = chosen[:N_SHOT]
    print(f'Few-shot examples: {[e.key for e in examples]}')

llm = create_provider(host='qwen')
print(f'Model: {llm}')
print(f'Generating attack path for {TARGET_CVE}...')

result = run_single(
    cve_id=TARGET_CVE,
    cve_description=entry.description,
    llm=llm,
    selected_oi=SELECTED_OI,
    few_shot_examples=examples
)

print('\n===== Generated PDDL =====')
print(result)

if SAVE_OUTPUT:
    out_dir = '../../experiments'
    os.makedirs(out_dir, exist_ok=True)
    shot_label = f'{N_SHOT}shot'
    out_file = os.path.join(out_dir, f'{TARGET_CVE}_qwen_manual_{shot_label}.pddl')
    with open(out_file, 'w') as f:
        f.write(result)
    print(f'Saved to: {out_file}')

Few-shot examples: ['CVE-2022-1471 / AP1']
Model: QwenProvider(model='qwen-plus')
Generating attack path for CVE-2025-66032...

===== Generated PDDL =====
(define (domain AED)
  (:requirements
    :adl
    :fluents
  )
  (:functions
    (total-cost)
    (version ?Software)
    (port ?Port)
  )
  (:types
    attacker - actor
    user - actor
    target-system
    software
    infrastructure
    cve-identifier
    exploit-technique - technique
    cli-command - command
    shell-context - context
    code-agent - agent
    read-only-validation - validation
    malicious-shell-command - command
    untrusted-content - content
    context-window - window
    command-execution-logic
  )
  (:constants
    CVE_2025_66032 - cve-identifier
    bypass-read-only-validation - exploit-technique
    inject-malicious-shell-command - exploit-technique
  )
  (:predicates
    (spoofing ?Target - target-system)
    (tampering ?Target - target-system)
    (repudiation ?Target - target-system)
    (informa